# 🇰🇬 Обучение своей модели AkylAi

Этот ноутбук дообучает открытую модель **Qwen2.5-3B** на датасете AkylAi (кыргызский язык, УК КР, ПДД, Конституция, история).

**Как запустить:**
1. Открой в Google Colab.
2. Меню → Среда выполнения → Сменить среду выполнения → **GPU (T4)**.
3. Нажимай на каждой ячейке ▶ по порядку.

Бесплатно. Обучение занимает ~30–60 минут.

## 1. Установка инструментов

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

## 2. Загрузка датасета из GitHub

In [ ]:
!git clone https://github.com/5005os/Cloud-Code.git
!python Cloud-Code/training/prepare_dataset.py
!wc -l Cloud-Code/training/akylai_dataset.jsonl

## 3. Загрузка базовой модели (Qwen2.5-3B)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Подготовка данных (формат чата)

In [ ]:
from datasets import load_dataset

SYSTEM = "Ты — AkylAi, кыргызский ИИ-ассистент. Отвечай точно про Кыргызстан (КР): кыргызский язык, законы КР, ПДД, Конституцию, историю и культуру."

dataset = load_dataset("json", data_files="Cloud-Code/training/akylai_dataset.jsonl", split="train")

def format_chat(ex):
    user = ex["instruction"]
    if ex.get("input"):
        user += "\n" + ex["input"]
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat)
print(dataset[0]["text"][:500])

## 5. Обучение (LoRA)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer.train()

## 6. Проверка — поговори со своей моделью

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)
    print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

ask("Как будет 'спасибо' на кыргызском?")
ask("Что предусматривает статья 122 УК КР?")
ask("Расскажи про Манас на кыргызском.")

## 7. Сохранить модель (LoRA + GGUF для локального запуска)

In [ ]:
# LoRA-адаптер (маленький)
model.save_pretrained("akylai_lora")
tokenizer.save_pretrained("akylai_lora")

# GGUF — чтобы запускать в Ollama / llama.cpp на своём компьютере
model.save_pretrained_gguf("akylai_model", tokenizer, quantization_method="q4_k_m")
print("Готово! Модель AkylAi сохранена. Скачай папку akylai_model (файл .gguf).")

## 🎉 Готово!

У тебя своя модель **AkylAi**. Дальше:
- Запустить локально через **Ollama**: `ollama create akylai -f Modelfile` (модель .gguf).
- Или поднять как API на сервере и подключить к сайту ittinamy.ru.
- Чтобы модель была умнее — пополняй `dataset/` и обучай снова.